In [1]:
import os

BASE_URL = f"http://localhost:8000/v1"

os.environ["BASE_URL"]    = BASE_URL
os.environ["OPENAI_API_KEY"] = "abc-123"   

print("Config set:", BASE_URL)

Config set: http://localhost:8000/v1


In [2]:
!pip uninstall -y pydantic-ai-slim openai pandas numpy 


Found existing installation: pydantic-ai-slim 1.107.0
Uninstalling pydantic-ai-slim-1.107.0:
  Successfully uninstalled pydantic-ai-slim-1.107.0
Found existing installation: openai 2.41.1
Uninstalling openai-2.41.1:
  Successfully uninstalled openai-2.41.1
Found existing installation: pandas 3.0.3
Uninstalling pandas-3.0.3:
  Successfully uninstalled pandas-3.0.3
Found existing installation: numpy 2.4.6
Uninstalling numpy-2.4.6:
  Successfully uninstalled numpy-2.4.6


In [4]:
 !pip install "pydantic-ai-slim[openai]" pandas scikit-learn

  Using cached pandas-3.0.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached pydantic_ai_slim-1.107.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached openai-2.41.1-py3-none-any.whl.metadata (32 kB)
  Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached pydantic_ai_slim-1.107.0-py3-none-any.whl (964 kB)
Using cached pandas-3.0.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 188.5 MB/s  0:00:00
Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
Using cached openai-2.41.1-py3-none-any.whl (1.4 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [pydantic-ai-slim][pydantic-ai-slim]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-py

In [5]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
print(os.environ.get("BASE_URL"))
provider = OpenAIProvider(
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

llm_30b = OpenAIChatModel("Qwen3-30B-A3B", provider=provider)

http://localhost:8000/v1


In [6]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

print("✅ Import success")

provider = OpenAIProvider(
    base_url="http://localhost:8000/v1",
    api_key="abc-123"
)

llm_30b = OpenAIChatModel(
    model_name="Qwen3-30B-A3B",
    provider=provider
)

✅ Import success


In [20]:
import pandas as pd
import numpy as np
import random

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, recall_score


user_df = pd.DataFrame({
    "user_id": [f"U{i}" for i in range(1000)],
    "account_age_days": np.random.randint(10, 1000, 1000)
})


device_df = pd.DataFrame({
    "device_id": [f"D{i}" for i in range(1000)],
    "user_id": np.random.choice(user_df["user_id"], 1000),
    "device_consistency": np.random.uniform(0.7, 1.0, 1000),
    "is_new_device": np.random.choice([0, 1], 1000)
})



behavior_df = pd.DataFrame({
    "user_id": user_df["user_id"],
    "behavior_score": np.random.uniform(0.6, 1.0, len(user_df))
})


#events_df = pd.DataFrame({
#    "event_id": [f"E{i}" for i in range(5000)],
#    "user_id": np.random.choice(user_df["user_id"], 5000),
#    "device_id": np.random.choice(device_df["device_id"], 5000),
#    "sim_changed_recently": np.random.choice([0, 1], 5000),
#    "geo_distance_from_last_login": np.random.uniform(0, 50, 5000)
#})


events_df = pd.DataFrame({
    "event_id": [f"E{i}" for i in range(5000)],
})

# 🔥 Pick valid device-user pairs
sampled_devices = device_df.sample(5000, replace=True).reset_index(drop=True)

events_df["user_id"] = sampled_devices["user_id"]
events_df["device_id"] = sampled_devices["device_id"]

events_df["sim_changed_recently"] = np.random.choice([0, 1], 5000)
events_df["geo_distance_from_last_login"] = np.random.uniform(0, 50, 5000)


def generate_labels(events_df):
    labels = []

    for _, row in events_df.iterrows():
        risk = (
            0.3 * row["sim_changed_recently"] +
            0.2 * (row["geo_distance_from_last_login"] > 20)
        )
        labels.append(1 if risk > 0.4 else 0)

    return pd.DataFrame({
        "event_id": events_df["event_id"],
        "is_fraud": labels
    })

labels_df = generate_labels(events_df)


df = events_df.merge(labels_df, on="event_id")
df = df.merge(device_df, on=["user_id", "device_id"], how="left")
df = df.merge(behavior_df, on="user_id", how="left")



MODEL_FEATURES = [
    "device_consistency",
    "is_new_device",
    "behavior_score",
    "sim_changed_recently",
    #"geo_distance_from_last_login"
]

X = df[features]
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

ml_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42
)

ml_model.fit(X_train, y_train)

y_prob = ml_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob > 0.3).astype(int)  # 🔥 tuned threshold

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

tn, fp, fn, tp = cm.ravel()

print(f"""
False Negatives (MOST IMPORTANT): {fn}
Recall: {recall_score(y_test, y_pred)}
ROC-AUC: {roc_auc_score(y_test, y_prob)}
""")



y_prob = ml_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob > 0.3).astype(int)  # 🔥 tuned threshold

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

tn, fp, fn, tp = cm.ravel()

print(f"""
False Negatives (MOST IMPORTANT): {fn}
Recall: {recall_score(y_test, y_pred)}
ROC-AUC: {roc_auc_score(y_test, y_prob)}
""")


def context_agent(user_id, device_id):
    df = device_df[
        (device_df["user_id"] == user_id) &
        (device_df["device_id"] == device_id)
    ]

    if df.empty:
        # 🔥 fallback (important for real systems)
        return {
            "device_consistency": 0.5,
            "is_new_device": 1
        }

    device = df.iloc[0]

    return {
        "device_consistency": device["device_consistency"],
        "is_new_device": device["is_new_device"]
    }

    
def telecom_agent(event):
    return {
        "sim_changed_recently": event["sim_changed_recently"],
        "geo_distance_from_last_login": event["geo_distance_from_last_login"]
    }


def behavior_agent(user_id):
    df = behavior_df[behavior_df["user_id"] == user_id]

    if df.empty:
        return {"behavior_score": 0.5}

    row = df.iloc[0]

    return {
        "behavior_score": row["behavior_score"]
    }

#def risk_agent(features):

#    df_input = pd.DataFrame([features])
#    prob = ml_model.predict_proba(df_input)[0][1]

    # 🔥 Rule overrides
#    if features["sim_changed_recently"] == 1:
#        prob = max(prob, 0.8)

#    if features["is_new_device"] == 1:
#        prob = max(prob, 0.5)

#    return {
#        "risk_score": round(prob, 2),
#        "risk_level": "high" if prob > 0.7 else "medium" if prob > 0.3 else "low"
#    }


def risk_agent(features):

    # ✅ Keep only trained features
    model_input = {k: features[k] for k in MODEL_FEATURES}

    df_input = pd.DataFrame([model_input])

    prob = ml_model.predict_proba(df_input)[0][1]

    # 🔥 Rule overrides (can still use extra signals)
    if features["sim_changed_recently"] == 1:
        prob = max(prob, 0.8)

    if features["is_new_device"] == 1:
        prob = max(prob, 0.5)

    return {
        "risk_score": round(prob, 2),
        "risk_level": "high" if prob > 0.7 else "medium" if prob > 0.3 else "low"
    }


from pydantic_ai import Agent

decision_agent = Agent(
    model=llm_30b,
    system_prompt="""
    Decide authentication:

    low → no OTP
    medium → step-up
    high → OTP

    Return JSON with decision + reason.
    """
)

#def generate_labels(events_df):
#    labels = []

#    for _, row in events_df.iterrows():
#        risk = (
#            0.3 * row["sim_changed_recently"] +
#            0.2 * (row["geo_distance_from_last_login"] > 20) +
#            np.random.uniform(0, 0.3)  # 🔥 noise
#        )
#        labels.append(1 if risk > 0.5 else 0)

#    return pd.DataFrame({
#        "event_id": events_df["event_id"],
#        "is_fraud": labels
#    })

def generate_labels(events_df):
    labels = []

    for _, row in events_df.iterrows():
        risk = (
            0.25 * row["sim_changed_recently"] +
            0.15 * (row["geo_distance_from_last_login"] > 20) +
            0.2 * random.choice([0, 1]) +   # 🔥 hidden factor
            np.random.uniform(0, 0.4)       # 🔥 noise
        )

        labels.append(1 if risk > 0.6 else 0)

    return pd.DataFrame({
        "event_id": events_df["event_id"],
        "is_fraud": labels
    })
    
async def run_auth_flow(event):

    user_id = event["user_id"]
    device_id = event["device_id"]

    features = {
        **context_agent(user_id, device_id),
        **telecom_agent(event),
        **behavior_agent(user_id)
    }

    risk = risk_agent(features)

    prompt = f"""
    Risk Score: {risk['risk_score']}
    Risk Level: {risk['risk_level']}
    Features: {features}
    """

    decision = await decision_agent.run(prompt)

    return {
        "features": features,
        "risk": risk,
        "decision": decision.output
    }

              precision    recall  f1-score   support

           0       0.94      0.77      0.85       695
           1       0.63      0.89      0.74       305

    accuracy                           0.81      1000
   macro avg       0.79      0.83      0.80      1000
weighted avg       0.85      0.81      0.82      1000

Confusion Matrix:
 [[538 157]
 [ 33 272]]

False Negatives (MOST IMPORTANT): 33
Recall: 0.8918032786885246
ROC-AUC: 0.8662365845028894

              precision    recall  f1-score   support

           0       0.94      0.77      0.85       695
           1       0.63      0.89      0.74       305

    accuracy                           0.81      1000
   macro avg       0.79      0.83      0.80      1000
weighted avg       0.85      0.81      0.82      1000

Confusion Matrix:
 [[538 157]
 [ 33 272]]

False Negatives (MOST IMPORTANT): 33
Recall: 0.8918032786885246
ROC-AUC: 0.8662365845028894



In [21]:
import numpy as np

thresholds = np.arange(0.1, 0.9, 0.05)

for t in thresholds:
    y_pred = (y_prob > t).astype(int)
    recall = recall_score(y_test, y_pred)
    print(f"Threshold: {t:.2f}, Recall: {recall:.3f}")

import pandas as pd

importance = pd.DataFrame({
    "feature": features,
    "importance": ml_model.feature_importances_
}).sort_values(by="importance", ascending=False)

print(importance)

sample_event = events_df.iloc[0]
result = await run_auth_flow(sample_event)
print(result)

Threshold: 0.10, Recall: 0.984
Threshold: 0.15, Recall: 0.967
Threshold: 0.20, Recall: 0.944
Threshold: 0.25, Recall: 0.911
Threshold: 0.30, Recall: 0.892
Threshold: 0.35, Recall: 0.862
Threshold: 0.40, Recall: 0.843
Threshold: 0.45, Recall: 0.820
Threshold: 0.50, Recall: 0.793
Threshold: 0.55, Recall: 0.754
Threshold: 0.60, Recall: 0.725
Threshold: 0.65, Recall: 0.692
Threshold: 0.70, Recall: 0.626
Threshold: 0.75, Recall: 0.528
Threshold: 0.80, Recall: 0.469
Threshold: 0.85, Recall: 0.410
                feature  importance
3  sim_changed_recently    0.692469
0    device_consistency    0.157743
2        behavior_score    0.144051
1         is_new_device    0.005737
{'features': {'device_consistency': np.float64(0.724089768017274), 'is_new_device': np.int64(1), 'sim_changed_recently': np.int64(0), 'geo_distance_from_last_login': np.float64(19.47131131571726), 'behavior_score': np.float64(0.9843594128630329)}, 'risk': {'risk_score': 0.5, 'risk_level': 'medium'}, 'decision': '\n\n{\n  "